# Results Dashboard

Live performance / CLV analysis for the strikeout-prop paper ledger. Reloads
`artifacts/odds_log/ledger.parquet` fresh on every run — re-run the notebook
(Kernel > Restart & Run All) any time to refresh.

Sections:
1. Overall performance
2. Last 7 days
3. Results vs. CLV (by day, by edge bin)
4. Side / book splits
5. K-bias / calibration by side
6. Kelly vs. flat sizing comparison
7. Bankroll curve
8. Rolling CLV trend

> Reminder: this is an exploratory pilot (see `docs/reference/market_clv_gates.md`).
> Nothing here should be read as proven edge until `n_clv >= 100` **and** the
> bootstrap CI on mean CLV excludes 0.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

%matplotlib inline

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "Python").exists():
    if (ROOT.parent / "src" / "Python").exists():
        ROOT = ROOT.parent.resolve()
    else:
        raise FileNotFoundError(f"Cannot find src/Python from cwd={Path.cwd()}")
sys.path.insert(0, str(ROOT / "src"))

from Python.market import DEFAULT_EDGE_FLOOR, bet_pnl, bootstrap_mean_ci
from Python.odds_ledger import LEDGER_PATH, dedupe_ledger_props, settled_bets

MIN_CLV_N = 100  # pre-registered skill-gate sample size (market_clv_gates.md)
EDGE_BIN_ORDER = ["<8% (no bet)", "8-12%", "12-16%", "16-20%", "20%+"]


def edge_bin_expr(col: str = "edge") -> pl.Expr:
    e = pl.col(col) * 100
    return (
        pl.when(e < 8).then(pl.lit("<8% (no bet)"))
        .when(e < 12).then(pl.lit("8-12%"))
        .when(e < 16).then(pl.lit("12-16%"))
        .when(e < 20).then(pl.lit("16-20%"))
        .otherwise(pl.lit("20%+"))
        .alias("edge_bin")
    )


def load_ledger() -> pl.DataFrame:
    if not LEDGER_PATH.exists():
        raise FileNotFoundError(f"No ledger at {LEDGER_PATH}. Run poll_odds.py --snapshot open first.")
    raw = pl.read_parquet(LEDGER_PATH)
    props = dedupe_ledger_props(raw)  # one row per (date, pitcher, line) - no DK+FD double count
    if "game_date" in props.columns:
        props = props.with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10).alias("game_date"))
    return props


full = load_ledger()
staked = full.filter((pl.col("status") == "settled") & (pl.col("stake") > 0)) if full.height else full
print(f"ledger={LEDGER_PATH}")
print(f"props (deduped)={full.height}  staked_settled={staked.height}")

## 1. Overall Performance

In [ ]:
def summarize(df: pl.DataFrame, label: str) -> dict:
    d = df.filter(pl.col("stake") > 0) if "stake" in df.columns else df
    n = d.height
    pnl = float(d["pnl"].sum()) if n and "pnl" in d.columns else 0.0
    staked_amt = float(d["stake"].sum()) if n and "stake" in d.columns else 0.0
    win_rate = float(d["result"].eq("win").mean()) if n and "result" in d.columns else float("nan")
    roi = (pnl / staked_amt * 100) if staked_amt else float("nan")
    clv_all = full.filter(pl.col("clv_pp").is_not_null())
    clvs = [float(x) for x in clv_all["clv_pp"].to_list()] if clv_all.height else []
    row = {"cohort": label, "n_bets": n, "pnl": round(pnl, 2), "staked": round(staked_amt, 2),
           "roi_pct": round(roi, 2), "win_rate": round(win_rate, 3), "n_clv": len(clvs)}
    if len(clvs) >= 5:
        m, lo, hi = bootstrap_mean_ci(clvs)
        row["mean_clv_pct"] = round(m * 100, 3)
        row["clv_ci_lo_pct"] = round(lo * 100, 3)
        row["clv_ci_hi_pct"] = round(hi * 100, 3)
        gate = "PASS" if lo > 0 and len(clvs) >= MIN_CLV_N else (
            "FAIL" if hi < 0 and len(clvs) >= MIN_CLV_N else "INCONCLUSIVE"
        )
        row["gate"] = gate
    return row

overall_row = summarize(staked, "overall (all-time)")
overall_df = pl.DataFrame([overall_row])
print(overall_df)
print(f"\nCLV skill gate: n_clv={overall_row['n_clv']} / {MIN_CLV_N} required "
      f"({min(100, overall_row['n_clv'] / MIN_CLV_N * 100):.0f}% of sample target)")

## 2. Last 7 Days

In [ ]:
if full.height and "game_date" in full.columns:
    from datetime import datetime, timedelta

    max_date = datetime.strptime(full["game_date"].max(), "%Y-%m-%d").date()
    cutoff = max_date - timedelta(days=6)
    last7 = staked.filter(
        pl.col("game_date").str.strptime(pl.Date, "%Y-%m-%d", strict=False) >= cutoff
    )

    daily = (
        last7.group_by("game_date")
        .agg(
            pl.len().alias("n"),
            pl.col("pnl").sum().alias("pnl"),
            pl.col("stake").sum().alias("staked"),
            pl.col("result").eq("win").mean().alias("win_rate"),
        )
        .with_columns((pl.col("pnl") / pl.col("staked") * 100).round(2).alias("roi_pct"))
        .sort("game_date")
    )
    print(f"Last 7 days (since {cutoff}):")
    print(daily)
    last7_row = summarize(last7, "last_7_days")
    print()
    print(pl.DataFrame([last7_row]))
else:
    print("No ledger rows yet.")

## 3. Results vs. CLV

Same-day PnL and CLV can diverge — a day can win money on variance while the
market still moved against the closing side (negative CLV), or vice versa.
CLV is the more forward-looking signal since it doesn't depend on a single
game's K outcome.

In [ ]:
clv_by_day = (
    full.filter(pl.col("clv_pp").is_not_null())
    .group_by("game_date")
    .agg(pl.col("clv_pp").mean().alias("mean_clv_pct") * 100, pl.len().alias("n_clv"))
    .sort("game_date")
)
pnl_by_day = (
    staked.group_by("game_date")
    .agg(pl.col("pnl").sum().alias("pnl"), pl.col("result").eq("win").mean().round(3).alias("win_rate"))
    .sort("game_date")
)
results_vs_clv = pnl_by_day.join(clv_by_day, on="game_date", how="full", coalesce=True).sort("game_date")
print("Results vs. CLV by day:")
print(results_vs_clv)

In [ ]:
staked_binned = staked.with_columns(edge_bin_expr())
edge_bin_tbl = (
    staked_binned.group_by("edge_bin")
    .agg(
        pl.len().alias("n"),
        pl.col("pnl").sum().round(2).alias("pnl"),
        pl.col("stake").sum().round(2).alias("staked"),
        pl.col("result").eq("win").mean().round(3).alias("win_rate"),
        pl.col("clv_pp").mean().alias("mean_clv_pct"),
        pl.col("clv_pp").count().alias("n_clv"),
    )
    .with_columns(
        (pl.col("pnl") / pl.col("staked") * 100).round(2).alias("roi_pct"),
        (pl.col("mean_clv_pct") * 100).round(3),
    )
    .with_columns(pl.col("edge_bin").cast(pl.Enum(EDGE_BIN_ORDER)))
    .sort("edge_bin")
)
print("Edge-bin performance (staked bets only, best-edge book per prop):")
print(edge_bin_tbl)

In [ ]:
clv_scatter_src = full.filter(pl.col("clv_pp").is_not_null() & pl.col("edge").is_not_null())
if clv_scatter_src.height:
    pdf = clv_scatter_src.select(
        (pl.col("edge") * 100).alias("edge_pct"),
        (pl.col("clv_pp") * 100).alias("clv_pct"),
        "side",
    ).to_pandas()

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for side, color in [("over", "tab:red"), ("under", "tab:blue")]:
        sub = pdf[pdf["side"] == side]
        ax.scatter(sub["edge_pct"], sub["clv_pct"], alpha=0.6, label=side, color=color, s=28)
    if len(pdf) >= 3:
        corr = pdf["edge_pct"].corr(pdf["clv_pct"])
        ax.set_title(f"CLV vs. model edge  (r={corr:+.3f}, n={len(pdf)})")
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.axvline(DEFAULT_EDGE_FLOOR * 100, color="gray", linewidth=0.8, linestyle=":")
    ax.set_xlabel("Model edge (%)")
    ax.set_ylabel("CLV (pp)")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("No CLV data yet.")

## 4. Side / Book Splits

In [ ]:
by_side = (
    staked.group_by("side")
    .agg(pl.len().alias("n"), pl.col("pnl").sum().round(2).alias("pnl"),
         pl.col("result").eq("win").mean().round(3).alias("win_rate"))
)
by_book = (
    staked.group_by("book")
    .agg(pl.len().alias("n"), pl.col("pnl").sum().round(2).alias("pnl"),
         pl.col("result").eq("win").mean().round(3).alias("win_rate"))
)
print("--- by side ---")
print(by_side)
print("\n--- by book ---")
print(by_book)

print("\n--- book x edge-bin (pnl) ---")
book_edge = (
    staked_binned.group_by(["book", "edge_bin"])
    .agg(pl.len().alias("n"), pl.col("pnl").sum().round(2).alias("pnl"))
    .with_columns(pl.col("edge_bin").cast(pl.Enum(EDGE_BIN_ORDER)))
    .sort(["book", "edge_bin"])
)
print(book_edge)

## 5. K-Bias / Calibration by Side

Joins each settled ticket back to its logged projection
(`artifacts/projection_log/projections.parquet`) to compare `expected_K`
against the actual `settle_value`. A positive bias means the model
over-projected strikeouts (favors `over` bets that then don't cash).

In [ ]:
PROJECTIONS_PATH = ROOT / "artifacts" / "projection_log" / "projections.parquet"

if PROJECTIONS_PATH.exists() and staked.height:
    proj = (
        pl.read_parquet(PROJECTIONS_PATH)
        .with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10))
        .select(["game_date", "player_name", "expected_K"])
        .unique(subset=["game_date", "player_name"], keep="first")
    )
    joined = staked.join(proj, on=["game_date", "player_name"], how="left").filter(
        pl.col("expected_K").is_not_null() & pl.col("settle_value").is_not_null()
    )
    if joined.height:
        bias = joined.with_columns((pl.col("expected_K") - pl.col("settle_value")).alias("k_bias"))
        bias_by_side = (
            bias.group_by("side")
            .agg(
                pl.len().alias("n"),
                pl.col("k_bias").mean().round(3).alias("mean_k_bias"),
                pl.col("p_model").mean().round(3).alias("mean_p_model"),
                pl.col("result").eq("win").mean().round(3).alias("actual_win_rate"),
            )
            .with_columns((pl.col("mean_p_model") - pl.col("actual_win_rate")).round(3).alias("calibration_gap"))
        )
        print("K-bias & calibration gap by side (positive k_bias = model over-projected K):")
        print(bias_by_side)
    else:
        print("No joined rows with both expected_K and settle_value yet.")
else:
    print("Missing projections log or no staked/settled bets yet.")

## 6. Kelly vs. Flat Sizing Comparison

Re-derives PnL for each settled/staked ticket under three alternate sizing
rules, using the ticket's **actual recorded open price** and real outcome
(`result`) — only the stake changes:

- **actual (fractional Kelly)**: what was really staked (`stake` column)
- **flat_1u**: every bet sized at exactly 1 unit (`unit_dollars`)
- **flat_1.5x_20plus**: 1u, except 1.5u when model edge >= 20%
- **flat_2x_20plus**: 1u, except 2.0u when model edge >= 20%

In [ ]:
def scenario_pnl(df: pl.DataFrame, boost_mult: float | None) -> pl.DataFrame:
    won = pl.col("result") == "win"
    if boost_mult is None:
        stake_expr = pl.col("unit_dollars")
    else:
        stake_expr = pl.when(pl.col("edge") >= 0.20).then(pl.col("unit_dollars") * boost_mult).otherwise(pl.col("unit_dollars"))
    dec = pl.when(pl.col("bet_price") > 0).then(pl.col("bet_price") / 100 + 1).otherwise(100 / (-pl.col("bet_price")) + 1)
    pnl_expr = pl.when(won).then(stake_expr * (dec - 1)).otherwise(-stake_expr)
    return df.with_columns(stake_expr.alias("_scn_stake"), pnl_expr.alias("_scn_pnl"))

scenarios = {
    "actual_kelly": None,
    "flat_1u": None,
    "flat_1.5x_20plus": 1.5,
    "flat_2x_20plus": 2.0,
}

rows = []
base = staked.filter(pl.col("unit_dollars").is_not_null() & pl.col("bet_price").is_not_null())
for name, boost in scenarios.items():
    if name == "actual_kelly":
        pnl_sum = float(base["pnl"].sum())
        stake_sum = float(base["stake"].sum())
    else:
        scn = scenario_pnl(base, boost)
        pnl_sum = float(scn["_scn_pnl"].sum())
        stake_sum = float(scn["_scn_stake"].sum())
    roi = pnl_sum / stake_sum * 100 if stake_sum else float("nan")
    rows.append({"scenario": name, "n": base.height, "total_staked": round(stake_sum, 2),
                 "total_pnl": round(pnl_sum, 2), "roi_pct": round(roi, 2)})

scenario_tbl = pl.DataFrame(rows)
print(scenario_tbl)
print("\nNote: flat scenarios keep the same 8% edge floor for bet selection - only the stake formula changes.")

## 7. Bankroll / Cumulative PnL Curve

In [ ]:
if base.height:
    sort_col = "closed_at_utc" if "closed_at_utc" in base.columns else "game_date"
    ordered = base.sort(["game_date", sort_col])
    pdf = ordered.select("game_date", "pnl").to_pandas()
    pdf["cum_actual"] = pdf["pnl"].cumsum()

    for name, boost in [("flat_1u", None), ("flat_1.5x_20plus", 1.5), ("flat_2x_20plus", 2.0)]:
        scn = scenario_pnl(ordered, boost).select("_scn_pnl").to_pandas()
        pdf[f"cum_{name}"] = scn["_scn_pnl"].cumsum()

    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = range(len(pdf))
    ax.plot(x, pdf["cum_actual"], label="actual (fractional Kelly)", linewidth=2)
    ax.plot(x, pdf["cum_flat_1u"], label="flat 1u", linestyle="--")
    ax.plot(x, pdf["cum_flat_1.5x_20plus"], label="flat 1u / 1.5u @ 20%+", linestyle=":")
    ax.plot(x, pdf["cum_flat_2x_20plus"], label="flat 1u / 2.0u @ 20%+", linestyle="-.")
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_xlabel("Bet # (chronological)")
    ax.set_ylabel("Cumulative PnL ($)")
    ax.set_title("Cumulative PnL: Kelly vs. flat sizing scenarios")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("No staked/settled bets yet.")

## 8. Rolling CLV Trend

Rolling mean CLV over the last 20 / 50 props (chronological), against the
zero line. Watch whether this drifts up/down as sample size grows toward the
`n_clv >= 100` gate.

In [ ]:
clv_ordered = full.filter(pl.col("clv_pp").is_not_null()).sort(["game_date", "closed_at_utc"] if "closed_at_utc" in full.columns else ["game_date"])
if clv_ordered.height >= 5:
    pdf = clv_ordered.select((pl.col("clv_pp") * 100).alias("clv_pct")).to_pandas()
    pdf["roll20"] = pdf["clv_pct"].rolling(20, min_periods=5).mean()
    pdf["roll50"] = pdf["clv_pct"].rolling(50, min_periods=10).mean()

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(pdf.index, pdf["roll20"], label="rolling 20-bet mean CLV")
    ax.plot(pdf.index, pdf["roll50"], label="rolling 50-bet mean CLV")
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.axvline(min(MIN_CLV_N, len(pdf)) - 1, color="tab:red", linewidth=0.8, linestyle=":", label=f"n={MIN_CLV_N} gate")
    ax.set_xlabel("Prop # (chronological, with CLV)")
    ax.set_ylabel("Rolling mean CLV (pp)")
    ax.set_title(f"Rolling CLV trend (n_clv={len(pdf)})")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("Not enough CLV data yet for a rolling trend (need >= 5).")

## 9. Opponent Quality & Home/Away Splits

`opp_lineup_k_vs_hand` is the opponent lineup's K rate specifically against a
pitcher throwing the same hand as the starter that game (i.e. it's already
handedness-aware) - binning on it directly tells us whether we're getting
paid for attacking favorable platoon matchups, independent of raw opponent
strikeout rate (`opp_lineup_k`).

In [ ]:
if PROJECTIONS_PATH.exists() and staked.height:
    opp_cols = [
        c
        for c in ("game_date", "player_name", "is_home", "opp_lineup_k", "opp_lineup_k_vs_hand")
        if c in pl.scan_parquet(PROJECTIONS_PATH).collect_schema().names()
    ]
    opp = (
        pl.read_parquet(PROJECTIONS_PATH)
        .with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10))
        .select(opp_cols)
        .unique(subset=["game_date", "player_name"], keep="first")
    )
    staked_opp = staked.join(opp, on=["game_date", "player_name"], how="left")

    if "is_home" in staked_opp.columns:
        home_away = (
            staked_opp.filter(pl.col("is_home").is_not_null())
            .group_by("is_home")
            .agg(
                pl.len().alias("n"),
                pl.col("pnl").sum().round(2).alias("pnl"),
                pl.col("result").eq("win").mean().round(3).alias("win_rate"),
                pl.col("clv_pp").mean().alias("mean_clv_pct"),
            )
            .with_columns((pl.col("mean_clv_pct") * 100).round(3))
            .sort("is_home")
        )
        print("--- home vs. away starter ---")
        print(home_away)

    if "opp_lineup_k_vs_hand" in staked_opp.columns:
        matchup = staked_opp.filter(pl.col("opp_lineup_k_vs_hand").is_not_null())
        if matchup.height >= 8:
            matchup = matchup.with_columns(
                pl.col("opp_lineup_k_vs_hand")
                .qcut(3, labels=["weak_matchup", "avg_matchup", "favorable_matchup"])
                .alias("matchup_tier")
            )
            matchup_tbl = (
                matchup.group_by("matchup_tier")
                .agg(
                    pl.len().alias("n"),
                    pl.col("pnl").sum().round(2).alias("pnl"),
                    pl.col("result").eq("win").mean().round(3).alias("win_rate"),
                    pl.col("clv_pp").mean().alias("mean_clv_pct"),
                    pl.col("opp_lineup_k_vs_hand").mean().round(4).alias("avg_opp_k_vs_hand"),
                )
                .with_columns((pl.col("mean_clv_pct") * 100).round(3))
                .sort("matchup_tier")
            )
            print("\n--- opponent-quality tiers (opp K rate vs. this pitcher's hand) ---")
            print(matchup_tbl)
        else:
            print("\nNeed >= 8 staked bets with opp_lineup_k_vs_hand to form tiers.")
else:
    print("Missing projections log or no staked/settled bets yet.")

## 10. Edge-Floor Sweep — whole settled universe, flat-1u vs. Kelly

**Why two ROI curves, not one.** Kelly-stake ROI (`sum(pnl) / sum(stake) %`) reflects your
*realized bankroll* — that's the official answer to "did I make money." But it
**mixes two effects into one variable**: bet selection (which bets fire) and
bet sizing (how big each bet is). Choose-Kelly means a single high-edge
leverage bet can dominate the sum; raising the floor then changes the
denominator as much as the numerator. That makes Kelly ROI a *bad* navigator
for setting an edge floor, which is fundamentally a per-bet yes/no decision.

Flat-1u-risked ROI fixes this: **every settled bet counts as exactly $1
risked regardless of edge or odds.** A win pays `(dec − 1) × $1`, a loss
costs `$1`. A −160 favorite loss costs $1, a +120 dog win pays $1.20;
favorites and dogs are penalized equally on a dollar-cost-of-loss basis.
Each bet contributes equally to the estimate, so the curve answers
"if I'd fired flat-1u on every settled bet with edge ≥ floor, what's my
ROI?" — that's the per-bet skill question the floor is supposed to control.

**The sweep covers the whole settled universe (incl. sub-8% observations).**
The previous version filtered to `stake > 0`, which silently excluded any
bet Kelly's `edge < 8%` rule declined — that's the very range we need to
inspect to judge whether 8% was set right. Now the flat-1u curve runs over
*all* graded-settled tickets including those Kelly passed on, so the sub-8%
behavior is visible.

**How to read the two curves together:**
- **Flat (red) line crosses 0**: the *per-bet skill break-even* floor.
  Should match DEFAULT_EDGE_FLOOR if the floor is set correctly.
- **Kelly (blue) line vs flat (red) line gap at any floor**: theKelly
  sizing premium/discount at that floor. If Kelly >> flat, Kelly is
  concentrating stake on a few winners (often small-n luck, not skill).
  If Kelly << flat, Kelly's high-edge stakes are dragging (variance, not
  wrong-side selection).
- **A high-floor "Kelly ROI explosion" with flat ROI going flat/negative
  is a variance mirage** — high-edge small-n buckets where Kelly happened
  to land a few big wins. Trust the flat line for floor-setting.

**Chart B (right):** finer CLV-by-edge-bin with bootstrap 95% CIs — the
skill oracle per bin. CLV positive across bins is the prerequisite for
proven skill; if CLV turns negative at the high edges while PnL goes up,
*the market says the high-edge PnL is luck, not skill.* Trust CLV over
PnL when small-n at high floors.

> Caveats:
> - The pre-registered skill gate (`n_clv ≥ 100`, bootstrap CI > 0) from
>   `docs/reference/market_clv_gates.md` still applies at the overall level.
>   Bin-level CIs are directional, not proof.
> - The 6 settled-but-missing-CLV tickets (SharpAPI gaps) skew slightly
>   toward late-posted / lower-liquidity props — a minor selection bias.

In [ ]:
import numpy as np

# Two parallel sources (both already deduped via `full = load_ledger()`):
#   * kelly_src  = tickets Kelly actually staked on (stake > 0). Used for the
#                  blue "Kelly PnL" curve so it reflects realized bankroll.
#   * flat_src   = ALL settled tickets, INCLUDING those Kelly passed on
#                  (stake == 0). Flat-1u-risked PnL is computed uniformly so
#                  we can sweep floor candidates the live sizing filter would
#                  have skipped. This is the curve that tests whether 8% was
#                  set correctly.
settled_all = settled_bets(full).filter(pl.col("edge").is_not_null())
kelly_src = settled_all.filter(pl.col("stake") > 0)
flat_src = settled_all

# Flat 1u-risked PnL: win -> +1 * (dec - 1); loss -> -1.
# dec-1 = bet_price/100 if bet_price > 0 else 100/(-bet_price).
# A -160 loss costs $1.00 (1u risked), a +120 win pays $1.20, all bets use
# the same $1 risked regardless of odds -- so favorites and dogs are
# penalized equally on a dollar-cost-of-loss basis.
flat_pnl_expr = (
    pl.when(pl.col("result") == "win")
    .then(
        pl.when(pl.col("bet_price") > 0)
        .then(pl.col("bet_price") / 100.0)
        .otherwise(100.0 / (-pl.col("bet_price")))
    )
    .otherwise(-1.0)
    .alias("flat_1u_pnl")
)
flat_src = flat_src.with_columns(flat_pnl_expr)

# ---- Edge-floor ROC sweep ----
FLOORS = np.arange(0, 26, 1)  # 0% .. 25% in 1-point steps
rows = []
for f in FLOORS:
    k = kelly_src.filter(pl.col("edge") * 100 >= f)
    fl = flat_src.filter(pl.col("edge") * 100 >= f)
    n_kelly = k.height
    n_flat = fl.height
    k_pnl = float(k["pnl"].sum()) if n_kelly else 0.0
    k_stake = float(k["stake"].sum()) if n_kelly else 0.0
    k_roi = (k_pnl / k_stake * 100) if k_stake else float("nan")
    fl_pnl = float(fl["flat_1u_pnl"].sum()) if n_flat else 0.0
    # Flat-1u risked = 1.0 per bet, so denominator = n_flat.
    fl_roi = (fl_pnl / n_flat * 100) if n_flat else float("nan")
    # Win/loss counts (flat universe) for an un-staked-skill sanity check.
    fl_wins = int(fl.filter(pl.col("result") == "win").height)
    clv_sub = flat_src.filter(pl.col("edge") * 100 >= f, pl.col("clv_pp").is_not_null())
    clvs = [float(x) for x in clv_sub["clv_pp"].to_list()] if clv_sub.height else []
    if len(clvs) >= 5:
        m, lo, hi = bootstrap_mean_ci(clvs)
        clv_lo, clv_hi, clv_mean = lo * 100, hi * 100, m * 100
    else:
        clv_mean, clv_lo, clv_hi = float("nan"), float("nan"), float("nan")
    rows.append(
        {
            "floor_pct": float(f),
            "n_kelly": n_kelly,
            "n_flat": n_flat,
            "n_clv": len(clvs),
            "kelly_pnl": k_pnl,
            "kelly_stake": k_stake,
            "kelly_roi_pct": k_roi,
            "flat_1u_pnl": fl_pnl,
            "flat_1u_roi_pct": fl_roi,
            "flat_wins": fl_wins,
            "flat_win_rate": (fl_wins / n_flat) if n_flat else float("nan"),
            "mean_clv_pp": clv_mean,
            "clv_lo": clv_lo,
            "clv_hi": clv_hi,
        }
    )

roc = pl.DataFrame(rows)
pd_roc = roc.to_pandas()

# ---- Chart B: finer CLV-by-edge-bin with 95% bootstrap CIs ----
FINER_BINS = [(0, 4), (4, 6), (6, 8), (8, 10), (10, 12), (12, 14),
              (14, 16), (16, 18), (18, 20), (20, 30)]
b_centers, b_clv, b_lo, b_hi, b_n = [], [], [], [], []
clv_src = settled_all.filter(pl.col("clv_pp").is_not_null())
for lo_e, hi_e in FINER_BINS:
    sub = clv_src.filter(
        (pl.col("edge") * 100 >= lo_e) & (pl.col("edge") * 100 < hi_e)
    )
    if sub.height == 0:
        continue
    clvs = [float(x) for x in sub["clv_pp"].to_list()]
    if len(clvs) >= 5:
        m, lo, hi = bootstrap_mean_ci(clvs)
    else:
        m = sum(clvs) / len(clvs) if clvs else float("nan")
        lo = hi = float("nan")
    b_centers.append((lo_e + hi_e) / 2)
    b_clv.append(m * 100)
    b_lo.append(lo * 100 if lo == lo else float("nan"))
    b_hi.append(hi * 100 if hi == hi else float("nan"))
    b_n.append(sub.height)

# ---- Render side-by-side ----
if pd_roc["n_flat"].max() or b_centers:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: two ROI curves vs edge floor
    ax1.plot(pd_roc["floor_pct"], pd_roc["flat_1u_roi_pct"], color="tab:red",
             lw=2, label="Flat 1u-risked ROI (all settled, incl. Kelly-passed)")
    ax1.plot(pd_roc["floor_pct"], pd_roc["kelly_roi_pct"], color="tab:blue",
             lw=2, label="Kelly-stake ROI (realized bankroll)")
    ax1.axhline(0, color="black", lw=0.7)
    ax1.axvline(DEFAULT_EDGE_FLOOR * 100, color="gray", lw=0.8, linestyle=":",
                label=f"current floor {DEFAULT_EDGE_FLOOR * 100:.0f}%")
    ax1.set_xlabel("Edge floor (%)  — sweep includes sub-8% bets")
    ax1.set_ylabel("Cumulative ROI at edge >= floor (%)")
    ax1.set_title("Edge-floor ROC: flat-1u (skill) vs. Kelly (realized)")
    ax1.legend(loc="upper left", fontsize=8)

    ax1b = ax1.twinx()
    ax1b.plot(pd_roc["floor_pct"], pd_roc["n_flat"], color="tab:orange",
              lw=1.4, linestyle="--", label="n bets (flat universe)")
    ax1b.plot(pd_roc["floor_pct"], pd_roc["n_kelly"], color="tab:purple",
              lw=1.0, linestyle=":", label="n bets (Kelly-staked)")
    ax1b.axhline(30, color="tab:orange", lw=0.5, linestyle=":", alpha=0.5)
    ax1b.set_ylabel("n bets (settled)", color="tab:orange")
    ax1b.tick_params(axis="y", labelcolor="tab:orange")
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax1b.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8)

    # Right: CLV by finer edge bin with CI error bars
    if b_centers:
        x = np.arange(len(b_centers))
        clv_arr = np.array(b_clv, dtype=float)
        lo_arr = np.array(b_lo, dtype=float)
        hi_arr = np.array(b_hi, dtype=float)

        ax2.bar(x - 0.2, clv_arr, width=0.4, color="tab:green",
                label="mean CLV (pp)")
        mask = ~(np.isnan(lo_arr) | np.isnan(hi_arr))
        if mask.any():
            ax2.errorbar(
                (x - 0.2)[mask], clv_arr[mask],
                yerr=[clv_arr[mask] - lo_arr[mask], hi_arr[mask] - clv_arr[mask]],
                fmt="none", ecolor="tab:green", capsize=3,
            )
        ax2.axhline(0, color="black", lw=0.7)
        ax2.set_xticks(x)
        ax2.set_xticklabels(
            [f"{lo_e}-{hi_e}%" for lo_e, hi_e in FINER_BINS if (lo_e+hi_e)/2 in b_centers],
            rotation=30,
        )
        ax2.set_ylabel("Mean CLV (pp) ± 95% bootstrap CI", color="tab:green")
        ax2.tick_params(axis="y", labelcolor="tab:green")
        ax2.set_title("CLV by finer edge bin — skill should be positive everywhere")

        ax2b = ax2.twinx()
        ax2b.bar(x + 0.2, b_n, width=0.4, color="lightgray",
                 alpha=0.6, label="n bets (with CLV)")
        ax2b.set_ylabel("n bets (with CLV)", color="gray")
        ax2b.tick_params(axis="y", labelcolor="gray")
        for xi, ni in zip(x, b_n):
            ax2b.text(xi + 0.2, ni, str(ni), ha="center", va="bottom", fontsize=7)
        h1, l1 = ax2.get_legend_handles_labels()
        h2, l2 = ax2b.get_legend_handles_labels()
        ax2.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=8)

    fig.tight_layout()
    plt.show()
else:
    print("Not enough settled bets with edge to compute the edge-floor charts.")

# Print the full ROC table. Use with_columns to round without dropping cols.
roc_display = roc.with_columns(
    pl.col("floor_pct").round(0),
    pl.col("kelly_roi_pct").round(2),
    pl.col("flat_1u_roi_pct").round(2),
    pl.col("flat_win_rate").round(3),
    pl.col("mean_clv_pp").round(3),
    pl.col("clv_lo").round(3),
    pl.col("clv_hi").round(3),
)
print("\n--- Edge-floor ROC: full dataset (incl. sub-8% observations) ---")
roc_display

## 11. CLV-vs-realized-win-rate calibration (reliability plot + z-test)

The 53.8% / 39.0% split from the 2026-08-06 read is the single best signal in
the ledger. Two ways this section stops that number from being a one-shot:

1. **Reliability plot**: bin settled bets by `clv_pp` decile, plot observed
   win rate per bin against the CLV-predicted win rate (Brier-style). A CLV
   oracle worth trusting should show wins clustering in the right tail and a
   monotonic curve above the 45° line. If the right tail is monotonic it's a
   skill oracle; if it's driven by two outlier bins the headline gap is
   variance, not edge.
2. **Two-proportion z-test head-to-head**: `two_proportion_z_test` (in
   `src/Python/skill_stats.py`) directly tests H0: `p_win(CLV≥+1.0pp) ==
   p_win(CLV<+1.0pp)`. As of 2026-08-06 the ledger gives `p_a=0.538, p_b=0.390`,
   `z≈1.84, p≈0.065` — **directionally strong but not yet significant at
   α=0.05**. State it precisely, not as settled.

The weekly reliability artifact at `artifacts/odds_log/clv_reliability.parquet`
(one row per clv_pp bin with observed vs expected win rate and n) is written
here so the calibration check is persistent, not a one-off.

In [ ]:
from Python.market import american_to_implied_prob, devig_two_way
from Python.skill_stats import two_proportion_z_test

RELIABILITY_PATH = LEDGER_PATH.parent / "clv_reliability.parquet"

settled_with_clv = (
    settled_bets(full)
    .filter(pl.col("clv_pp").is_not_null() & pl.col("result").is_not_null())
    if not full.is_empty() else full
)
if not settled_with_clv.is_empty() and settled_with_clv.height >= 10:
    # ---- Reliability: bin by clv_pp decile, observed vs CLV-predicted win rate ----
    src = settled_with_clv.with_columns(
        (pl.col("clv_pp") * 100.0).alias("clv_pp_100"),
        (pl.col("result") == "win").alias("won"),
    ).sort("clv_pp_100")

    # Decile bins on clv_pp (qcut handles ties via the smallest-value rule).
    # For n<30 we fall back to 5 bins so each cell has enough mass.
    n_bins = 10 if src.height >= 60 else max(2, src.height // 6)
    try:
        binned = src.with_columns(
            pl.col("clv_pp_100").qcut(n_bins, allow_duplicates=True).alias("bin")
        )
    except Exception:
        # qcut can fail on heavy ties at low n; fall back to equal-width cut.
        lo, hi = float(src["clv_pp_100"].min()), float(src["clv_pp_100"].max())
        edges = [lo + (hi - lo) * i / n_bins for i in range(n_bins + 1)]
        binned = src.with_columns(
            pl.col("clv_pp_100").cut(edges).alias("bin")
        )

    reliability = (
        binned.group_by("bin")
        .agg(
            pl.len().alias("n"),
            pl.col("won").mean().alias("observed_win_rate"),
            pl.col("clv_pp_100").mean().alias("mean_clv_pp"),
            (pl.col("p_market") + pl.col("clv_pp")).mean().alias("expected_win_rate"),
        )
        .sort("mean_clv_pp")
    )

    # Persist as the weekly calibration artifact (docs/reference/market_clv_gates.md).
    reliability.write_parquet(RELIABILITY_PATH)
    print(f"clv_reliability.parquet written: {reliability.height} bins, n={settled_with_clv.height}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(reliability.height)
    obs = reliability["observed_win_rate"].to_list()
    exp = reliability["expected_win_rate"].to_list()
    ns = reliability["n"].to_list()

    ax1.plot([0, 1], [0, 1], color="gray", lw=0.8, linestyle="--", label="perfect calibration")
    ax1.scatter(exp, obs, s=[max(20, n * 4) for n in ns], alpha=0.7, color="tab:blue")
    for i, (e, o, n) in enumerate(zip(exp, obs, ns)):
        ax1.annotate(f"n={n}", (e, o), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax1.set_xlabel("CLV-predicted win rate")
    ax1.set_ylabel("Observed win rate")
    ax1.set_title(f"CLV reliability (deciles, n={settled_with_clv.height})")
    ax1.legend(loc="upper left", fontsize=8)

    # ---- z-test head-to-head: CLV>=+1.0pp vs CLV<+1.0pp ----
    sig_pos = settled_with_clv.filter(pl.col("clv_pp") >= 0.01)
    sig_neg = settled_with_clv.filter(pl.col("clv_pp") < 0.01)
    s_a = int(sig_pos.filter(pl.col("result") == "win").height)
    n_a = sig_pos.height
    s_b = int(sig_neg.filter(pl.col("result") == "win").height)
    n_b = sig_neg.height

    if n_a > 0 and n_b > 0:
        zr = two_proportion_z_test(s_a, n_a, s_b, n_b)
        ax2.bar(
            ["CLV≥+1.0pp", "CLV<+1.0pp"],
            [zr["p_a"], zr["p_b"]],
            color=["tab:green", "tab:red"],
            alpha=0.7,
        )
        ax2.axhline(0.524, color="gray", lw=0.8, linestyle=":", label="break-even (0.524)")
        for i, (p, nn) in enumerate(zip([zr["p_a"], zr["p_b"]], [n_a, n_b])):
            ax2.annotate(f"{p:.1%}\nn={nn}", (i, p), xytext=(0, 5), textcoords="offset points", ha="center", fontsize=9)
        title = (f"Win-rate head-to-head: z={zr['z']:+.2f}, p={zr['p_two_sided']:.3f} "
                 f"({'SIGNIFICANT' if zr['p_two_sided'] < 0.05 else 'not yet at α=0.05'})")
        ax2.set_title(title, fontsize=10)
        ax2.set_ylabel("Win rate")
        ax2.set_ylim(0, max(0.7, max(zr["p_a"], zr["p_b"]) + 0.1))
        ax2.legend(loc="upper right", fontsize=8)
        print(f"z-test: z={zr['z']:+.3f}, p_two_sided={zr['p_two_sided']:.4f}")
        print(f"  CLV>=+1.0pp: {zr['p_a']:.3f} (n={n_a}, wins={s_a})")
        print(f"  CLV< +1.0pp: {zr['p_b']:.3f} (n={n_b}, wins={s_b})")

    fig.tight_layout()
    plt.show()
    print("\n--- clv_reliability.parquet (decile-level calibration) ---")
    print(reliability)
else:
    print(f"Not enough settled CLV bets for the reliability plot (have {settled_with_clv.height if not settled_with_clv.is_empty() else 0}, need >= 10).")

## 12. Band-discrete flat-1u panel (the noise the cumulative sweep hides)

Section 10 draws a cumulative edge≥floor curve, which **always** looks smoother
than the underlying data. The discrete band view below shows `flat-1u ROI`,
`mean CLV`, `<f,f+1)` n-count and 95% CI bars per *band*. It's deliberately
ugly — that's the point. The two questions it exposes:

1. Is the floor sitting on a real edge or on cumulative noise? (The `[6,9)`
   band with `median CLV ≈ 0` and only 13/25 wins is the kind of segment the
   cumulative curve hides — you can't see that from a CI bar.)
2. Are the bands above the active floor monotonic, or is the apparent edge at
   floor=12% pulled by one hot band?

This panel would have stopped me from trusting the cumulative smoothness in
the first place; it stops it from baiting another floor move.

In [ ]:
# Section 12: discrete-band flat-1u panel — the noise the cumulative sweep hides.
#
# Section 10 plots cumulative edge>=floor curves, which always look smoother
# than the underlying bins. Here we slice into 1pp-wide discrete edge bands and
# show per-band `n`, flat-1u ROI, and mean CLV with a **BCa** CI (percentile
# bootstrap is biased at the per-band n≈10-40 range; see
# `bootstrap_bca_ci` in src/Python/skill_stats.py). The point is to expose any
# band sitting on cumulative noise — e.g. an `[f, f+1)` cell where n is large
# but median CLV drifts to zero — before that bait gets used for a floor move.
#
# Bands with n<5 get a degenerate CI bar (no CI drawn); that's intentional —
# we don't pretend the data is more confident than it is.

from Python.skill_stats import bootstrap_bca_ci

# Re-use `flat_src` (all settled tickets with edge, incl. Kelly-passed) and
# `flat_pnl_expr` defined in Section 10. Only require non-null edge here.
band_src = (
    settled_bets(full)
    .filter(pl.col("edge").is_not_null())
    .with_columns(flat_pnl_expr)
)

# Discrete bands: 1pp wide from 0..25. Same coverage as Section 10's FINER_BINS
# logic but uniform spacing so every band is comparable to its neighbors.
BAND_EDGES = list(range(0, 26))  # [0,1), [1,2), ... [24,25)

b_rows = []
for lo_e in BAND_EDGES:
    hi_e = lo_e + 1
    sub = band_src.filter(
        (pl.col("edge") * 100 >= lo_e) & (pl.col("edge") * 100 < hi_e)
    )
    n = sub.height
    if n == 0:
        continue

    pnl = float(sub["flat_1u_pnl"].sum())
    roi = (pnl / n * 100) if n else float("nan")
    wins = int(sub.filter(pl.col("result") == "win").height)
    win_rate = (wins / n) if n else float("nan")

    clv_sub = sub.filter(pl.col("clv_pp").is_not_null())
    clvs = [float(x) for x in clv_sub["clv_pp"].to_list()] if clv_sub.height else []
    if len(clvs) >= 5:
        m, lo, hi = bootstrap_bca_ci(clvs, n_boot=2000, seed=101 + lo_e)
        clv_mean, clv_lo, clv_hi = m * 100, lo * 100, hi * 100
    elif clvs:
        clv_mean = (sum(clvs) / len(clvs)) * 100
        clv_lo = clv_hi = float("nan")
    else:
        clv_mean = clv_lo = clv_hi = float("nan")

    b_rows.append({
        "band": f"[{lo_e},{hi_e})",
        "lo": lo_e,
        "n": n,
        "flat_roi_pct": roi,
        "flat_wins": wins,
        "flat_win_rate": win_rate,
        "mean_clv_pp": clv_mean,
        "clv_lo": clv_lo,
        "clv_hi": clv_hi,
        "n_clv": len(clvs),
    })

bands = pl.DataFrame(b_rows)

if bands.is_empty():
    print("No settled bets with edge to band.")
else:
    print(f"{bands.height} discrete bands, n_total={int(bands['n'].sum())}")
    pd_b = bands.to_pandas()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    x = np.arange(bands.height)
    floor_band_idx = next((i for i, lo in enumerate(bands["lo"].to_list())
                           if lo >= DEFAULT_EDGE_FLOOR * 100), None)

    # Left: flat-1u ROI per band + win-rate twin
    ax1.bar(x, pd_b["flat_roi_pct"], width=0.6,
            color=["tab:green" if r >= 0 else "tab:red"
                   for r in pd_b["flat_roi_pct"]], alpha=0.75)
    ax1.axhline(0, color="black", lw=0.7)
    if floor_band_idx is not None:
        ax1.axvspan(floor_band_idx - 0.5, bands.height - 0.5, color="gray",
                    alpha=0.08, label=f"bands >= active floor ({DEFAULT_EDGE_FLOOR * 100:.0f}%)")
        ax1.legend(loc="upper right", fontsize=8)
    ax1.set_xticks(x)
    ax1.set_xticklabels(bands["band"].to_list(), rotation=45, ha="right", fontsize=8)
    ax1.set_xlabel("Edge band (%)")
    ax1.set_ylabel("Flat 1u ROI (%)")
    ax1.set_title("Discrete-band flat-1u ROI (the noise the cumulative sweep hides)")
    for xi, (roi, ni) in enumerate(zip(pd_b["flat_roi_pct"], bands["n"].to_list())):
        ax1.annotate(f"n={ni}", (xi, roi), ha="center",
                     textcoords="offset points", xytext=(0, 4 if roi >= 0 else -10),
                     fontsize=7)

    ax1b = ax1.twinx()
    ax1b.plot(x, pd_b["flat_win_rate"], color="tab:purple", lw=1.5, marker="o",
              ms=4, label="win rate")
    ax1b.axhline(0.524, color="gray", lw=0.8, linestyle=":", label="break-even 0.524")
    ax1b.set_ylabel("Win rate", color="tab:purple")
    ax1b.tick_params(axis="y", labelcolor="tab:purple")
    ax1b.set_ylim(0, 1)
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax1b.get_legend_handles_labels()
    ax1b.legend(h2, l2, loc="upper left", fontsize=8)

    # Right: mean CLV per band with BCa CIs + n(CLV) twin
    clv_mean = np.array(pd_b["mean_clv_pp"], dtype=float)
    lo_arr = np.array(pd_b["clv_lo"], dtype=float)
    hi_arr = np.array(pd_b["clv_hi"], dtype=float)
    ax2.bar(x, clv_mean, width=0.6, color="tab:blue", alpha=0.75)
    mask = ~(np.isnan(lo_arr) | np.isnan(hi_arr))
    if mask.any():
        ax2.errorbar(
            x[mask], clv_mean[mask],
            yerr=[clv_mean[mask] - lo_arr[mask], hi_arr[mask] - clv_mean[mask]],
            fmt="none", ecolor="tab:blue", capsize=4, lw=1.0,
        )
    ax2.axhline(0, color="black", lw=0.7)
    ax2.set_xticks(x)
    ax2.set_xticklabels(bands["band"].to_list(), rotation=45, ha="right", fontsize=8)
    ax2.set_xlabel("Edge band (%)")
    ax2.set_ylabel("Mean CLV (pp) with BCa 95% CI")
    ax2.set_title("Discrete-band CLV (BCa — percentile bias corrected at low n)")
    for xi, (m, nl) in enumerate(zip(pd_b["mean_clv_pp"], bands["n_clv"].to_list())):
        ax2.annotate(f"nCLV={nl}", (xi, m), ha="center",
                     textcoords="offset points", xytext=(0, -12 if (not np.isnan(m) and m < 0) else 4),
                     fontsize=7, color="tab:blue")

    ax2b = ax2.twinx()
    ax2b.bar(x, bands["n_clv"].to_list(), width=0.2, color="lightgray",
             alpha=0.5, label="n(CLV)")
    ax2b.set_ylabel("n (with CLV)", color="gray")
    ax2b.tick_params(axis="y", labelcolor="gray")

    fig.tight_layout()
    plt.show()

    # ---- Persist the discrete band table ----
    BANDS_PATH = LEDGER_PATH.parent / "edge_band_discrete.parquet"
    bands.write_parquet(BANDS_PATH)
    print(f"\n--- edge_band_discrete.parquet (-> {BANDS_PATH.name}) ---")
    print(bands)
    print("\nFlags:")
    # 1) Below-floor bands with positive ROI (the recursive-floor-rediscovery bait)
    bait = bands.filter(pl.col("flat_roi_pct") > 0,
                        pl.col("lo") < DEFAULT_EDGE_FLOOR * 100)
    if bait.height:
        for b in bait["band"].to_list():
            print(f"  - {b}: positive ROI below the active floor ({DEFAULT_EDGE_FLOOR * 100:.0f}%) — DO NOT use this band's ROI to argue for lowering the floor")
    else:
        print("  - no sub-floor band has positive cumulative ROI")

    # 2) Above-floor bands where CI excludes zero (the only bands you can trust)
    trust = bands.filter(
        (pl.col("clv_lo") > 0) | (pl.col("clv_hi") < 0)
    ).filter(pl.col("lo") >= DEFAULT_EDGE_FLOOR * 100)
    if trust.height:
        for b, lo, hi in zip(trust["band"].to_list(), trust["clv_lo"].to_list(),
                             trust["clv_hi"].to_list()):
            print(f"  - {b}: BCa CLV CI [{lo:+.2f}, {hi:+.2f}] EXCLUDES zero -> trustworthy signal")
    else:
        print("  - no above-floor band has a CLV CI that excludes zero (need more n)")

## 13. Rolling 30-bet CLV (±2 SE ribbon) — day-stability check

The cumulative CLV mean hides *when* the edge showed up. My own note flagged
week-over-week ROI swings of **−36.88% / +40.81%** — this section is the
fastest way to see whether the CLV signal is 2–3 hot days or genuinely steady.

We emit, in chronological order, a rolling 30-bet mean of `clv_pp` with a
normal-approximation ±2 SE ribbon (`rolling_stat_with_se`). If the rolling
mean spends most of its time **above zero and outside the SE ribbon**, the
signal is steady. If it spikes on a handful of days and the rest of the
window is flat-to-negative, the headline CLV is a date artifact and the
n_clv ≥ 150 gate should hold — not move.

In [ ]:
# Section 13: rolling 30-bet CLV with ±2 SE ribbon.
#
# `rolling_stat_with_se` (src/Python/skill_stats.py) returns, per bet in
# chronological order, the rolling mean and a ±se_scale × SE ribbon via the
# normal approximation (sd / sqrt(n)). We plot the rolling mean against the
# ribbon so we can eyeball: (a) does the mean spend most of its life above
# zero, and (b) does it actually clear the ribbon (i.e. its SE), or just sit
# inside it.

from Python.skill_stats import rolling_stat_with_se

_sb = settled_bets(full)
_chrono = next(c for c in ("closed_at_utc", "logged_at_utc", "game_date") if c in _sb.columns)
roll_src = _sb.filter(pl.col("clv_pp").is_not_null()).sort(_chrono)

clv_vals = [float(x) for x in roll_src["clv_pp"].to_list()]
roll = rolling_stat_with_se(clv_vals, window=30, se_scale=2.0)

if len(clv_vals) < 30:
    print(f"Need >=30 settled CLV bets for a rolling-30 window (have {len(clv_vals)}).")
else:
    import numpy as np
    # rolling_stat_with_se returns None for lo/hi while n<2; coerce to NaN for plotting.
    def _to_float_or_nan(v):
        return float(v) if v is not None else float("nan")
    means = np.array([_to_float_or_nan(r["mean"]) for r in roll]) * 100  # pp -> pp*100
    lo = np.array([_to_float_or_nan(r["lo"]) for r in roll]) * 100
    hi = np.array([_to_float_or_nan(r["hi"]) for r in roll]) * 100
    xsi = np.array([r["idx"] for r in roll], dtype=float)

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.fill_between(xsi, lo, hi, color="tab:blue", alpha=0.18, label="±2 SE ribbon")
    ax.plot(xsi, means, color="tab:blue", lw=1.8, label="rolling 30-bet CLV mean")
    ax.axhline(0, color="black", lw=0.7)
    ax.axhline(DEFAULT_EDGE_FLOOR * 100 / 2, color="gray", lw=0.7, linestyle=":",
               label=f"half-active-floor ({DEFAULT_EDGE_FLOOR * 100 / 2:.1f}pp)")
    ax.set_xlabel("Settled-bet index (chronological)")
    ax.set_ylabel("CLV (pp)")
    ax.set_title(f"Rolling 30-bet CLV — is the signal steady or a date artifact? (n={len(clv_vals)})")
    ax.legend(loc="upper right", fontsize=8)

    # How often is the rolling mean *outside* the ribbon (above the upper bound)?
    above = int(sum(1 for r in roll if r["mean"] is not None and r["hi"] is not None and r["mean"] > r["hi"]))
    below_zero = int(sum(1 for r in roll if r["mean"] is not None and r["mean"] > 0))
    total = int(sum(1 for r in roll if r["mean"] is not None))
    print(f"Rolling windows above the SE ribbon: {above}/{total}")
    print(f"Rolling windows with positive mean: {below_zero}/{total} ({below_zero/total:.0%})")
    frac_above = above / total if total else 0.0
    if frac_above >= 0.5:
        print(f"  -> {above}/{total} windows ({frac_above:.0%}) clear the SE ribbon: signal is STEADY.")
    elif below_zero / total >= 0.7 and frac_above < 0.25:
        print(f"  -> only {above}/{total} ({frac_above:.0%}) windows clear the SE ribbon, though most are positive: signal is real but under-powered; keep the n>=150 gate.")
    else:
        print(f"  -> only {above}/{total} ({frac_above:.0%}) windows clear the SE ribbon: signal looks date-driven, do NOT use it to justify a floor move.")
    plt.show()

## 14. Stake-weighted CLV — the metric that actually governs bankroll

Equal-weighted CLV (`sum(clv)/n`) is the right *skill* metric, but production
risk is Kelly-sized: bigger-stake bets dominate realized returns. So before
any Kelly-scaling decision the headline claim should be **stake-weighted**:
`sum(clv × stake) / sum(stake)`. If the stake-weighted CLV diverges from the
equal-weighted one in the *negative* direction, the big-Kelly bets are costing
us — the sizing filter is amplifying bad CLV even as the small bets look fine.

We use `stake_weighted_bootstrap_ci` (BCa) for the confidence interval, again
because n is small and percentile intervals are biased here.

In [ ]:
# Section 14: stake-weighted CLV — sum(clv * stake) / sum(stake), with BCa CI.
#
# The operating skill claim under Kelly sizing has to be stake-weighted.
# `stake_weighted_bootstrap_ci` (src/Python/skill_stats.py) gives a BCa CI on
# that weighted mean. We also show equal-weighted CLV side-by-side so the
# divergence (or lack of it) is visible.

from Python.skill_stats import bootstrap_bca_ci, stake_weighted_bootstrap_ci

sw_src = settled_bets(full).filter(
    pl.col("clv_pp").is_not_null()
    & pl.col("stake").is_not_null()
    & (pl.col("stake") > 0)  # only bets Kelly actually staked on
)

if sw_src.height < 10:
    print(f"Not enough staked+CLV settled bets for stake-weighted CI (have {sw_src.height}, need >=10).")
else:
    clvs = [float(x) for x in sw_src["clv_pp"].to_list()]
    stakes = [float(x) for x in sw_src["stake"].to_list()]

    # Equal-weighted BCa CI (production uses n_boot=5000 per skill_stats default).
    ew_mean, ew_lo, ew_hi = bootstrap_bca_ci(clvs, n_boot=5000, seed=42)
    # Stake-weighted BCa CI (same defaults).
    sw_mean, sw_lo, sw_hi = stake_weighted_bootstrap_ci(clvs, stakes, n_boot=5000, seed=43)
    naive_weighted = sum(c * s for c, s in zip(clvs, stakes)) / sum(stakes)

    print(f"n (staked+CLV)         : {len(clvs)}")
    print(f"sum(stake)             : {sum(stakes):.2f}")
    print(f"equal-weighted CLV     : {ew_mean*100:+.3f}pp  [{ew_lo*100:+.3f}, {ew_hi*100:+.3f}]  (BCa 95%)")
    print(f"stake-weighted CLV     : {sw_mean*100:+.3f}pp  [{sw_lo*100:+.3f}, {sw_hi*100:+.3f}]  (BCa 95%)")
    print(f"  naive (sanity)       : {naive_weighted*100:+.3f}pp")
    diff = sw_mean - ew_mean
    print(f"diff (stake-equal)     : {diff*100:+.3f}pp  {'(bigger bets dragged CLV up)' if diff > 0 else '(bigger bets dragged CLV DOWN — sizing filter is amplifying bad CLV)'}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Left: bars with BCa CI error bars
    labels = ["equal-weighted", "stake-weighted"]
    means = [ew_mean * 100, sw_mean * 100]
    los = [ew_lo * 100, sw_lo * 100]
    his = [ew_hi * 100, sw_hi * 100]
    errs = [[means[i] - los[i] for i in range(2)],
            [his[i] - means[i] for i in range(2)]]
    colors = ["tab:gray", "tab:blue"]
    ax1.bar(labels, means, color=colors, alpha=0.75, width=0.5)
    ax1.errorbar(labels, means, yerr=errs, fmt="none", ecolor="black", capsize=6, lw=1.2)
    ax1.axhline(0, color="black", lw=0.7)
    ax1.set_ylabel("Mean CLV (pp) with BCa 95% CI")
    ax1.set_title(f"Equal vs. stake-weighted CLV (n={len(clvs)})")
    for i, (m, lo, hi) in enumerate(zip(means, los, his)):
        ax1.annotate(f"{m:+.2f}pp\n[{lo:+.2f}, {hi:+.2f}]", (i, m),
                     textcoords="offset points", xytext=(0, 14 if m >= 0 else -28),
                     ha="center", fontsize=9)

    # Right: cumulative stake-weighted CLV over chronological bets
    _chrono = next(c for c in ("closed_at_utc", "logged_at_utc", "game_date") if c in sw_src.columns)
    sw_src_sorted = sw_src.sort(_chrono)
    cs_clv = sw_src_sorted["clv_pp"].to_list()
    cs_stake = sw_src_sorted["stake"].to_list()
    cum_clv, cum_stake, running = [], [], []
    rc, rs = 0.0, 0.0
    for c, s in zip(cs_clv, cs_stake):
        rc += float(c) * float(s)
        rs += float(s)
        running.append((rc / rs) * 100 if rs else float("nan"))
    ax2.plot(range(1, len(running) + 1), running, color="tab:blue", lw=1.8)
    ax2.axhline(0, color="black", lw=0.7)
    # Final equal-weighted line for reference
    ew_running = []
    rc = 0.0
    for c in cs_clv:
        rc += float(c)
        ew_running.append((rc / len(ew_running) if len(ew_running) else 0.0) if False else (rc / (len(ew_running)+1)) * 100)
    ax2.plot(range(1, len(ew_running) + 1), ew_running, color="tab:gray", lw=1.0,
             linestyle="--", label="equal-weighted (running)")
    ax2.set_xlabel("Settled-bet index (chronological)")
    ax2.set_ylabel("Running mean CLV (pp)")
    ax2.set_title("Cumulative CLV — stake- vs. equal-weighted (does the big bet help or hurt?)")
    ax2.legend(loc="lower right", fontsize=9)
    plt.show()

## 15. Pseudo-ROC — CLV as a classifier for `result == win`

Sweep the threshold `t` from very low (predict all bets win) to very high
(predict no bets win), and for each `t` plot:

- **TPR** = P(predicted positive | actually positive) where "positive" = a
  winning bet, predicted = `clv_pp >= t`.
- **FPR** = P(predicted positive | actually negative) where "negative" =
  a losing bet.

A skill-free classifier sits on the 45° diagonal; a real skill oracle bulges
**up-and-left** (high TPR at low FPR). The AUC summarizes the whole curve in
one number. If `clv_pp` is a real skill oracle it should beat `AUC=0.5`. The
+1.0pp threshold from Section 11 is marked so you can read its TPR/FPR
position directly off the curve.

In [ ]:
# Section 15: pseudo-ROC for clv_pp as a classifier of result==win.
#
# Treat "predict win" as `clv_pp >= t`. Sweep t and compute:
#   TPR = (# winning bets with clv >= t) / (# winning bets)
#   FPR = (# losing  bets with clv >= t) / (# losing  bets)
# The classifier is well-defined iff every bet has a clv_pp and a result. The
# AUC is the trapezoidal integral of TPR vs FPR; 0.5 = no skill, >0.5 = the
# market drift aligns with outcomes.

roc_src = settled_bets(full).filter(
    pl.col("clv_pp").is_not_null() & pl.col("result").is_not_null()
).with_columns(
    (pl.col("result") == "win").alias("won")
)

if roc_src.height < 20:
    print(f"Not enough settled+CLV bets for pseudo-ROC (have {roc_src.height}, need >=20).")
else:
    total_w = int(roc_src.filter(pl.col("won")).height)
    total_l = roc_src.height - total_w
    if total_w == 0 or total_l == 0:
        print(f"Cannot build ROC curve with degenerate classes (wins={total_w}, losses={total_l}).")
    else:
        clvs = roc_src["clv_pp"].to_list()
        wons = roc_src["won"].to_list()

        thresholds = sorted(set(clvs), reverse=True)
        # Prepend a high threshold that classifies everything as negative.
        roc_points = [(1.0, 1.0)]  # (FPR, TPR) at the lowest threshold (predict all positive)
        # Actually build from highest threshold down: at the high end, no true positive.
        # We'll iterate thresholds over sorted unique clv_pp descending; at each we
        # count wins & losses with clv >= t.
        all_points = sorted(zip(clvs, wons), key=lambda pair: pair[0], reverse=True)
        cum_tp = cum_fp = 0
        fpr_tpr = [(0.0, 0.0)]  # threshold above max clv -> predict nothing
        for i, (c, won) in enumerate(all_points):
            if won:
                cum_tp += 1
            else:
                cum_fp += 1
            fpr_tpr.append((cum_fp / total_l, cum_tp / total_w))

        # AUC via trapezoidal integration (FPR is monotone non-decreasing).
        auc = 0.0
        for i in range(1, len(fpr_tpr)):
            x0, y0 = fpr_tpr[i - 1]
            x1, y1 = fpr_tpr[i]
            auc += (x1 - x0) * (y0 + y1) / 2.0

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.plot([p[0] for p in fpr_tpr], [p[1] for p in fpr_tpr],
                color="tab:blue", lw=2, label=f"CLV ROC (AUC={auc:.3f})")
        ax.plot([0, 1], [0, 1], color="gray", lw=0.8, linestyle="--",
                label="no-skill diagonal (AUC=0.5)")

        # Mark the +1.0pp threshold position (Section 11's headline).
        wins_ge = int(sum(1 for c, w in zip(clvs, wons) if c >= 0.01 and w))
        loss_ge = int(sum(1 for c, w in zip(clvs, wons) if c >= 0.01 and not w))
        tpr_t = wins_ge / total_w if total_w else 0.0
        fpr_t = loss_ge / total_l if total_l else 0.0
        ax.scatter([fpr_t], [tpr_t], color="tab:red", s=80, zorder=5,
                   label=f"t=+1.0pp (TPR={tpr_t:.2f}, FPR={fpr_t:.2f})")
        ax.set_xlabel("False Positive Rate (predict win | actually loss)")
        ax.set_ylabel("True Positive Rate (predict win | actually win)")
        ax.set_title(f"Pseudo-ROC — CLV as a classifier of result=win  (n={roc_src.height})")
        ax.legend(loc="lower right", fontsize=8)
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.05)
        print(f"AUC = {auc:.3f}  ({'BEATS no-skill' if auc > 0.5 else 'is AT OR BELOW no-skill' if auc <= 0.5 else ''}; 0.5 = coin flip, 0.6+ = real skill)")
        print(f"+1.0pp threshold (Section 11 split): TPR={tpr_t:.3f}, FPR={fpr_t:.3f}")
        if auc >= 0.6:
            print("  -> AUC >= 0.6: CLV is a real skill oracle for result=win.")
        elif auc > 0.5:
            print("  -> AUC in 0.5-0.6: signal is real but weak; the n>=150 gate is appropriate.")
        else:
            print("  -> AUC <= 0.5: CLV does NOT meaningfully order win/loss outcome in this ledger.")
        plt.show()

## 16. BCa-CLV sweep + pre-registered next-50-bet checkpoint

Two of the cheap-fix ask items in one place:

1. **BCa-CLV sweep**: replace the percentile CIs in Section 10's edge-floor
   sweep with `bootstrap_bca_ci`. Percentile intervals are biased at low n
   (~30-80 bets per floor), and the Section 10 sweep *specifically* reached the
   floor≥20% band where n is small. So Section 10's headline number for that
   band is currently over-confident — BCa fixes that. The resulting sweep stays
   in `artifacts/odds_log/clv_floor_bca.parquet` so the n_clv≥150 gate uses
   the same interval method the dashboard reports.

2. **Pre-registered next-50-bet checkpoint**: the actual fix for "recursive
   floor-rediscovery" (the disease the assistant named last session). We commit,
   up front, to scoring the *next 50 settled bets at the 12% floor blindly* —
   the universe at decision time is recorded here so we can later assert the
   50 bets scored were really the next-50-as-of-today, not a re-fitted slice.
   Decision criteria frozen here:

   - **Mean CLV ≥ +0.30pp AND win-rate ≥ 0.54** → keep floor at 12%, double
     the KB-class stake size (escalate).
   - **Mean CLV < +0.30pp OR win-rate < 0.524 (break-even)** → revert to ¼-Kelly
     sizing at the active floor; do NOT move the floor based on those 50 bets.
   - All other outcomes → no change; re-evaluate at n_clv = 200.

   The 50-bet scoring window itself doesn't run yet — that's the future audit
   step. What's frozen here is the rule and the universes-as-of date.

In [ ]:
# Section 16a: BCa-CLV sweep (replacing percentile on the floor >= 20% band).
#
# Mirrors Section 10's sweep but uses `bootstrap_bca_ci` so the headline CLV in
# the floor >= 14..20% bands isn't over-confident from a percentile bootstrap.
# The new table (clv_floor_bca.parquet) is the authoritative CLV-vs-floor
# artifact. Section 10 stays percentile for back-compat; this is the upgrade.

from Python.skill_stats import bootstrap_bca_ci
import datetime as _dt

floor_src = settled_bets(full).filter(pl.col("edge").is_not_null())
FLOORS = np.arange(0, 26, 1)
bca_rows = []
for f in FLOORS:
    sub = floor_src.filter((pl.col("edge") * 100 >= f) & pl.col("clv_pp").is_not_null())
    n = sub.height
    if n == 0:
        continue
    clvs = [float(x) for x in sub["clv_pp"].to_list()]
    if n >= 5:
        m, lo, hi = bootstrap_bca_ci(clvs, n_boot=5000, seed=200 + f)
        clv_mean, clv_lo, clv_hi = m * 100, lo * 100, hi * 100
    else:
        clv_mean = (sum(clvs) / len(clvs)) * 100
        clv_lo = clv_hi = float("nan")
    clv_subset = sub.filter(pl.col("result").is_not_null())
    wins = int(clv_subset.filter(pl.col("result") == "win").height)
    win_rate = (wins / clv_subset.height) if clv_subset.height else float("nan")
    bca_rows.append({
        "floor_pct": float(f),
        "n_clv": n,
        "clv_mean_pp": clv_mean,
        "clv_lo": clv_lo,
        "clv_hi": clv_hi,
        "ci_excludes_zero": (not (clv_lo != clv_lo)) and (clv_lo > 0 or clv_hi < 0),
        "settled_at_this_floor": clv_subset.height,
        "win_rate": win_rate,
    })

bca_sweep = pl.DataFrame(bca_rows)

BCA_PATH = LEDGER_PATH.parent / "clv_floor_bca.parquet"
bca_sweep.write_parquet(BCA_PATH)

if bca_sweep.is_empty():
    print("No settled CLV bets to sweep.")
else:
    print(f"BCa-CLV sweep written -> {BCA_PATH.name}  ({bca_sweep.height} floor levels)")
    print()
    print(bca_sweep)
    active_idx = next((i for i, f in enumerate(bca_sweep["floor_pct"].to_list())
                      if f >= DEFAULT_EDGE_FLOOR * 100), None)
    if active_idx is not None:
        row = bca_sweep.row(active_idx, named=True)
        print(f"\n--- At the active floor ({DEFAULT_EDGE_FLOOR*100:.0f}%) ---")
        print(f"n_clv                : {row['n_clv']}")
        print(f"BCa CLV mean         : {row['clv_mean_pp']:+.3f}pp  [{row['clv_lo']:+.3f}, {row['clv_hi']:+.3f}]")
        print(f"BCa CI excludes zero : {row['ci_excludes_zero']}")
        print(f"win-rate              : {row['win_rate']:.3f}  (break-even 0.524)")
        print(f"n_clv >= 150 gate    : {'MET' if row['n_clv'] >= 150 else 'not met ({n_clv}/150)'.replace('{n_clv}', str(row['n_clv']))}")
    # Visualize BCa vs Section 10's percentile CLV means if both exist.
    if not roc.is_empty() and not bca_sweep.is_empty():
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(bca_sweep["floor_pct"], bca_sweep["clv_mean_pp"], color="tab:blue", lw=2,
                label="BCa mean CLV (this section)")
        # Per-row CI bands
        lo_arr = np.array([x if x == x else np.nan for x in bca_sweep["clv_lo"].to_list()])
        hi_arr = np.array([x if x == x else np.nan for x in bca_sweep["clv_hi"].to_list()])
        ax.fill_between(bca_sweep["floor_pct"], lo_arr, hi_arr, color="tab:blue", alpha=0.18,
                        label="BCa 95% CI")
        # Section 10's percentile CLV means were stored as 'mean_clv_pp'
        if "mean_clv_pp" in roc.columns and not roc.filter(pl.col("n_clv") > 0).is_empty():
            roc_nonzero = roc.filter(pl.col("n_clv") > 0)
            ax.plot(roc_nonzero["floor_pct"], roc_nonzero["mean_clv_pp"], color="tab:gray", lw=1.2,
                    linestyle="--", label="percentile mean CLV (Section 10, biased at low n)")
        ax.axhline(0, color="black", lw=0.7)
        ax.axvline(DEFAULT_EDGE_FLOOR * 100, color="black", lw=0.7, linestyle=":",
                   label=f"active floor {DEFAULT_EDGE_FLOOR*100:.0f}%")
        ax.set_xlabel("Edge floor (%)")
        ax.set_ylabel("Mean CLV (pp) with 95% CI")
        ax.set_title("BCa vs. percentile CLV by edge floor — BCa is the authoritative sweep")
        ax.legend(loc="upper left", fontsize=8)
        plt.show()

## 16b. Pre-registered checkpoint — freeze the next-50-bet universe now

The cell below doesn't run any analysis. It **records** which bets are
*already settled* so the next run can assert the "next 50 bets" are bets that
were unsettled *as of now*. The pre-registration is stored at
`artifacts/odds_log/next_50_checkpoint.json` and includes the timestamp, the
current floor, the active KB size, and the SHA-256 of the ledger file so we
can later prove the audit universe was frozen at this decision point.

In [ ]:
# Section 16b — pre-register the next-50-bet checkpoint.
#
# We persist a JSON with: timestamp, active floor, KB-class stake size, SHA-256
# of the ledger, count of bets in each result-status state (settled/unsettled).
# Future audits read this file to assert the "next 50 settled" were chosen from
# the bets unsettled *as of this snapshot*, not a re-fitted window.

import json
import hashlib

# Pre-registered decision rule (see markdown above). This is *frozen* in the
# JSON so changes to this code in the future can't silently mutate the rule.
PREREG_RULE = {
    "version": 1,
    "active_floor_pct": float(DEFAULT_EDGE_FLOOR * 100),
    "next_n_bets": 50,
    "criteria": {
        "keep_floor_and_escalate_stake": {
            "mean_clv_pp_min": 0.30,
            "win_rate_min": 0.54,
        },
        "revert_to_quarter_kelly": {
            "mean_clv_pp_lt": 0.30,
            "win_rate_lt": 0.524,
        },
        "no_change": "all other outcomes — re-evaluate at n_clv=200",
    },
    "stopping_rule": {
        "gate": "n_clv >= 150 at floor >= 12%, evaluated on live-ledger reruns",
        "eval_at": ["BCa CLV CI on these n_clv", "win-rate vs 0.524 break-even"],
    },
}

settled_ledger = full
ledger_path = LEDGER_PATH

def _file_sha256(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()

# Compute the universe-as-of snapshot.
# `result_status_counts` walks the *whole* ledger; only bets with
# `result in {'win','loss'}` count as "settled" — `null` == unsettled.
# `ledger_height` is persisted so a future audit can assert the "next 50" were
# chosen from `n_unsettled` frozen at this point.
result_status_counts = (
    settled_ledger.with_columns(
        pl.when(pl.col("result") == "win").then(pl.lit("win"))
        .when(pl.col("result") == "loss").then(pl.lit("loss"))
        .when(pl.col("result").is_null()).then(pl.lit("unsettled"))
        .otherwise(pl.lit("other")).alias("result_class")
    )
    .group_by("result_class").agg(pl.len().alias("n")).sort("result_class")
)
universe = {row["result_class"]: row["n"] for row in result_status_counts.to_dicts()}

checkpoint = {
    "frozen_at_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "ledger_sha256": _file_sha256(ledger_path),
    "ledger_path": str(ledger_path),
    "ledger_height": int(settled_ledger.height),
    "universe_by_result_class": universe,
    "prereg_rule": PREREG_RULE,
}

CHECKPOINT_PATH = ledger_path.parent / "next_50_checkpoint.json"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH.write_text(json.dumps(checkpoint, indent=2), encoding="utf-8")

print(f"Pre-registration checkpoint written -> {CHECKPOINT_PATH.name}")
print(json.dumps(checkpoint, indent=2))